# Red Rising explainer — full-quality render on Kaggle

**Before running:**
1. Upload `Red_Rising_Project.zip` as a Kaggle **Dataset** (Kaggle unzips it automatically).
2. In this notebook: **Add Input** → your dataset.
3. **Settings → Accelerator: None** (CPU). A GPU does not speed this renderer up.
4. **Run All**. The finished film appears at the bottom as a download link (and in the *Output* tab).

Takes roughly 15–30 minutes on Kaggle's 4 CPU cores.

In [ ]:
import glob, os, shutil, subprocess, time
from concurrent.futures import ThreadPoolExecutor, as_completed

hits = glob.glob('/kaggle/input/**/vox.py', recursive=True)
assert hits, 'Project not found: add the Red_Rising_Project dataset with "Add Input".'
src = os.path.dirname(hits[0])
work = '/kaggle/working/vox'
if not os.path.exists(work):
    shutil.copytree(src, work)
os.chdir(work)
os.makedirs('out', exist_ok=True)
print('project:', work)
print(subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
!pip -q install pillow numpy scipy

In [ ]:
# Quality: CRF 12 is visually lossless. Use CRF 16 + 'medium' for a faster render.
os.environ.update(VOX_CRF='12', VOX_PRESET='slow', VOX_ABR='320k')
ACTS = {1: 'Act1_The_Mines', 2: 'Act2_The_Author', 3: 'Act3_A_World_Ruled_by_Color', 4: 'Act4_Eo_and_the_Lie',
        5: 'Act5_The_Institute', 6: 'Act6_Rome_and_the_Cost', 7: 'Act7_Allies_and_Enemies', 8: 'Act8_Legacy'}
PARALLEL = os.cpu_count() or 4

def render(n):
    t0 = time.time()
    r = subprocess.run(['python', f'act{n}_vox.py', 'video', f'out/{ACTS[n]}.mp4'], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'Act {n} failed:\n' + r.stderr[-3000:])
    return n, time.time() - t0

start = time.time()
with ThreadPoolExecutor(PARALLEL) as pool:
    jobs = [pool.submit(render, n) for n in (8, 7, 1, 6, 5, 4, 2, 3)]   # longest acts first
    for j in as_completed(jobs):
        n, secs = j.result()
        print(f'act {n} done in {secs/60:.1f} min  (total {(time.time()-start)/60:.1f} min)')

In [ ]:
with open('out/acts.txt', 'w') as f:
    for n in range(1, 9):
        f.write(f"file '{os.path.abspath('out/' + ACTS[n] + '.mp4')}'\n")
subprocess.run(['ffmpeg', '-y', '-v', 'error', '-f', 'concat', '-safe', '0', '-i', 'out/acts.txt',
                '-c', 'copy', '-movflags', '+faststart', '/kaggle/working/Red_Rising_Full.mp4'], check=True)
dur = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0',
                      '/kaggle/working/Red_Rising_Full.mp4'], capture_output=True, text=True).stdout.strip()
print(f'Red_Rising_Full.mp4: {float(dur):.1f} s, {os.path.getsize("/kaggle/working/Red_Rising_Full.mp4")/1e6:.0f} MB')

from IPython.display import FileLink
FileLink('/kaggle/working/Red_Rising_Full.mp4')